> **Learning-log notebook (kept as written).** Scratch work from building this project step by step.
> It predates the final cleanup, so some imports use the old module names (`Transformer_LM`, `training_together`, `AdamW.py`, `train_bpe_prallel`),
> and data paths point to local files that are not in this repo. Very large cell outputs were cleared. The code in `cs336_basics/` is the maintained version.


In [114]:
import numpy as np
import torch
import torch.nn as nn
from einops import einsum
class Linear(nn.Module):
    def __init__(self, in_features, out_features, device = None, dtype = None):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.device = device
        self.dtype = dtype
        linear_init = torch.normal(
            mean = 0,  std = 2 / (in_features+out_features) , size= (out_features, in_features)
            , requires_grad=True)
        linear_init = linear_init.clip(-3, 3)
        self.W = nn.Parameter(linear_init, requires_grad=True) 

    def forward(self, x: torch.Tensor):
        # X (in_feature, batch)
        output = einsum(self.W, x, "out in, in batch -> out batch")
        return output 

In [116]:
linear = Linear(4, 3)


# 🟢 Easy: Implement Softmax

Implement the **Softmax** function from scratch.

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

### Signature
```python
def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    ...
```

### Rules
- Do **NOT** use `torch.softmax`, `F.softmax`, or `torch.nn.Softmax`
- Must be **numerically stable** (hint: subtract `max` before `exp`)


In [33]:
def my_softmax(x:torch.Tensor, dim:int=-1) -> torch.Tensor:
    max_x = torch.max(x, dim = dim, keepdim=True).values
    numerator = torch.exp(x - max_x)
    denominator = numerator.sum(dim = dim, keepdim=True)
    return numerator / denominator

In [34]:
x = torch.tensor([1.0, 2.0, 3.0])
print("Output:", my_softmax(x, dim=-1))
print("Sum:   ", my_softmax(x, dim=-1).sum())  # should be ~1.0
print("Ref:   ", torch.softmax(x, dim=-1))

Output: tensor([0.0900, 0.2447, 0.6652])
Sum:    tensor(1.)
Ref:    tensor([0.0900, 0.2447, 0.6652])


# 🟡 Medium: Simple Linear Layer

Implement a fully-connected linear layer: **y = xW^T + b**

### Signature
```python
class SimpleLinear:
    def __init__(self, in_features: int, out_features: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- `self.weight`: shape `(out_features, in_features)`, init with `randn * (1/√in_features)`
- `self.bias`: shape `(out_features,)`, init as zeros
- Both must have `requires_grad=True`
- `forward(x)` computes `x @ W^T + b`
- Do **NOT** use `torch.nn.Linear`

In [31]:
import math

class SimpleLinear:
    def __init__(self, in_features: int, out_features: int):
        # pass  # Initialize weight and bias
        self.weight = torch.normal(mean = 0,
         std = 1/ math.sqrt(in_features), size= (out_features, in_features), requires_grad=True) 
        self.bias = torch.zeros((out_features, ), requires_grad=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # pass  # Compute y = x @ W^T + b
        return torch.matmul(x, self.weight.transpose(-1, -2)) + self.bias

In [32]:
# 🧪 Debug
layer = SimpleLinear(8, 4)
print("W shape:", layer.weight.shape)  # should be (4, 8)
print("b shape:", layer.bias.shape)    # should be (4,)

x = torch.randn(2, 8)
y = layer.forward(x)
print("Output shape:", y.shape)        # should be (2, 4)

W shape: torch.Size([4, 8])
b shape: torch.Size([4])
Output shape: torch.Size([2, 4])


# 🟡 Medium: Implement LayerNorm

Implement **Layer Normalization** from scratch.

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\mu$ and $\sigma^2$ are computed over the **last dimension**.

### Signature
```python
def my_layer_norm(
    x: torch.Tensor,      # input
    gamma: torch.Tensor,   # scale (same size as last dim)
    beta: torch.Tensor,    # shift (same size as last dim)
    eps: float = 1e-5
) -> torch.Tensor:
    ...
```

### Rules
- Do **NOT** use `F.layer_norm` or `torch.nn.LayerNorm`
- Normalize over the last dimension only
- Must support autograd

In [ ]:
def my_layer_norm(x, gamma, beta, eps=1e-5):
    # pass  # Replace this
    mu = torch.mean(x, dim = -1, keepdim=True)
    var = torch.var(x, dim = -1, keepdim=True, unbiased = False)
    return gamma * (x - mu) / torch.sqrt(var + eps) + beta




Your output mean: tensor([-5.9605e-08,  0.0000e+00])
Your output std:  tensor([1.0690, 1.0690])
Match ref?       True


In [36]:
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)

out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)

print("Your output mean:", out.mean(dim=-1))   # should be ~0
print("Your output std: ", out.std(dim=-1))     # should be ~1
print("Match ref?      ", torch.allclose(out, ref, atol=1e-4))

Your output mean: tensor([-2.2352e-08,  0.0000e+00])
Your output std:  tensor([1.0690, 1.0690])
Match ref?       True


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 Medium: Implement BatchNorm

Implement **Batch Normalization** with both **training** and **inference** behavior.

In training mode, use **batch statistics** and update running estimates:

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

where $\mu_B$ and $\sigma_B^2$ are the mean and variance computed **across the batch** (dim=0).

In inference mode, use the provided **running mean/var** instead of current batch stats.

### Signature
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — normalize each feature across all samples in the batch
    # running_mean, running_var: updated in-place during training; used as-is during inference
```

### Rules
- Do **NOT** use `F.batch_norm`, `nn.BatchNorm1d`, etc.
- Compute batch mean and variance over `dim=0` with `unbiased=False`
- Update running stats like PyTorch: `running = (1 - momentum) * running + momentum * batch_stat`
- Use `running_mean` / `running_var` for inference when `training=False`
- Must support autograd w.r.t. `x`, `gamma`, `beta` (running statistics should be treated as buffers, not parameters requiring gradients)

In [66]:
def my_batch_norm(
        x,
        gamma, 
        beta,
        running_mean,
        running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    if training:
        batch_mean = torch.mean(x, dim = 0, keepdim=True)
        batch_var = torch.var(x, dim = 0, keepdim=True, unbiased = False)
        # Use [:] to update the tensors in-place
        running_mean[:] = momentum*batch_mean + (1-momentum)* running_mean
        running_var[:] = batch_var * momentum +(1-momentum) * running_var
        BN = gamma * (x - batch_mean) / torch.sqrt(batch_var + eps) + beta

    else:
        batch_mean = running_mean
        batch_var = running_var

        BN = gamma * (x - batch_mean) / torch.sqrt(batch_var + eps) + beta
    return BN


In [67]:
# 🧪 Debug
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# Running stats typically live on the same device and shape as features
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training mode: uses batch stats and updates running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out_train.shape)
print("[Train] Column means:", out_train.mean(dim=0))   # should be ~0
print("[Train] Column stds: ", out_train.std(dim=0))    # should be ~1
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference mode: uses running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Output shape:", out_eval.shape)

[Train] Output shape: torch.Size([8, 4])
[Train] Column means: tensor([-2.9802e-08, -2.2352e-08,  0.0000e+00,  2.9802e-08])
[Train] Column stds:  tensor([1.0690, 1.0690, 1.0690, 1.0690])
Updated running_mean: tensor([-0.0451, -0.0054, -0.0870, -0.0021])
Updated running_var: tensor([0.9815, 1.1644, 0.9354, 0.9815])
[Eval] Output shape: torch.Size([8, 4])


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/08_rmsnorm.ipynb)

# 🟡 Medium: Implement RMSNorm

Implement **Root Mean Square Layer Normalization** — the normalization used in LLaMA, Gemma, etc.

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot w, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}$$

### Signature
```python
def rms_norm(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    # Normalize over the last dimension. No mean subtraction (unlike LayerNorm).
```

### Rules
- Do **NOT** use any built-in norm layers
- Normalize over `dim=-1`
- Must support autograd

In [93]:

def rms_norm(
        x, 
        weight, 
        eps : float = 1e-6
):
    sum_sq = torch.mean(x**2, dim = -1, keepdim=True) + eps
    rms = torch.sqrt(sum_sq)
    rms_norm =( weight.reshape((1, x.shape[-1]))*x) / rms
    return rms_norm

In [94]:
# 🧪 Debug
x = torch.randn(2, 8)
w = torch.ones(8)

out = rms_norm(x, w)
print("Output shape:", out.shape)
print("RMS of output:", out.pow(2).mean(dim=-1).sqrt())  # should be ~1

Output shape: torch.Size([2, 8])
RMS of output: tensor([1.0000, 1.0000])


# 🔴 Hard: Softmax Attention

Implement the core attention mechanism used in Transformers.

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Signature
```python
def scaled_dot_product_attention(
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
) -> torch.Tensor:   # (batch, seq_q, d_v)
    ...
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- You **may** use `torch.softmax` and `torch.bmm`
- Must support autograd
- Must handle cross-attention (seq_q ≠ seq_k)

In [56]:
from einops import einsum
def scaled_dot_product_attention(Q, K, V):
    batch, sq, dk = Q.shape
    score = einsum(Q, K.transpose(-2, -1), "batch sq dk, batch dk sk-> batch sq sk")
    # score = torch.matmul(Q, K.transpose(-2, -1))
    score = score / math.sqrt(dk)
    score = torch.softmax(score, dim = -1)
    # output = einsum(score, V, "batch sq sk, batch sk dv -> batch sq dv")
    output = torch.matmul(score, V)
    return output

In [57]:
# 🧪 Debug
torch.manual_seed(42)
Q = torch.randn(2, 4, 8)
K = torch.randn(2, 4, 8)
V = torch.randn(2, 4, 8)

out = scaled_dot_product_attention(Q, K, V)
print("Output shape:", out.shape)          # should be (2, 4, 8)
print("Has NaN?    ", torch.isnan(out).any().item())  # should be False
print("Has Inf?    ", torch.isinf(out).any().item())  # should be False

# Cross-attention: seq_q != seq_k
Q2 = torch.randn(1, 3, 16)
K2 = torch.randn(1, 5, 16)
V2 = torch.randn(1, 5, 32)
out2 = scaled_dot_product_attention(Q2, K2, V2)
print("Cross-attn shape:", out2.shape)     # should be (1, 3, 32)


Output shape: torch.Size([2, 4, 8])
Has NaN?     False
Has Inf?     False
Cross-attn shape: torch.Size([1, 3, 32])


# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [59]:
import torch.nn as nn

class MultiHeadAttention:
    def __init__(self, d_model:int, num_heads:int):
        self.d_model = d_model
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        # Q: (B, seq_q, d_model), K/V: (B, seq_k, d_model)
        self.W_q = nn.Linear(d_model, d_model, bias = False)
        self.W_k = nn.Linear(d_model, d_model, bias = False)
        self.W_v = nn.Linear(d_model, d_model, bias = False)
        self.W_o = nn.Linear(d_model, d_model, bias = False)

    def forward(self, Q, K, V):
        B, seq_q, d_model = Q.shape
        _, seq_k, _ = K.shape
        QW = self.W_q(Q) # B, seq_q, d_model
        QW = QW.reshape(B, seq_q, self.num_heads, self.d_k).transpose(2, 1) # B, num_heads, seq_q, d_k
        KW = self.W_k(K).reshape(B, seq_k, self.num_heads, self.d_k).transpose(2, 1) # B, num_heads, seq_k, d_k
        VW = self.W_v(V).reshape(B, seq_k, self.num_heads, self.d_k).transpose(2, 1)


        QKT = torch.matmul(QW, KW.transpose(-2, -1)) / math.sqrt(self.d_k) # B, num_heads, seq_q, seq_k
        score = torch.softmax(QKT, dim = -1)
        output = torch.matmul(score, VW) # B, num_heads, seq_q, d_k
        output = output.transpose(1, 2).reshape(B, seq_q, self.d_model)
        return self.W_o(output) # B, seq_q, d_model
    

        


In [60]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [102]:
def causal_attention(Q, K, V):
    """
    Q: torch.Tensor,  # (batch, seq_q, d_k)
    K: torch.Tensor,  # (batch, seq_k, d_k)
    V: torch.Tensor,  # (batch, seq_k, d_v)
    """
    batch, seq_q, d = Q.shape
    seq_k = K.shape[1]
    scaled_att_weights = torch.matmul(Q, K.transpose(-2, -1))  / math.sqrt(d) # batch, seq_q, seq_k
    casual_mask = torch.triu(
        torch.ones((seq_q, seq_k), dtype=torch.bool),
        diagonal=1
    )
    causal_att_weights = scaled_att_weights.masked_fill_(casual_mask, -float('inf'))
    causal_att_score = torch.softmax(causal_att_weights, dim = -1)
    output = torch.matmul(causal_att_score, V)
    return output



In [104]:
print(torch.triu(
    torch.ones((2, 4), dtype=torch.bool),
    diagonal=1
))
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

tensor([[False,  True,  True,  True],
        [False, False,  True,  True]])
Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [60]:
print(theta_mat.shape)
positions = torch.tensor([3,6,5,2,1], dtype = torch.int)
print(positions)
selected = cos[positions, :]


torch.Size([10, 20])
tensor([3, 6, 5, 2, 1], dtype=torch.int32)


In [ ]:
def rotation_mat(pos, cos, sin):
    diag_mat = torch.diag(cos[pos, :].repeat_interleave(2))
    sin_vec = sin[pos, :].repeat_interleave(2)
    sin_vec[1::2]=0
    off_diag_mat = torch.diag(sin_vec, diagonal=1)
    Ri = diag_mat - off_diag_mat + off_diag_mat.T
    return Ri

In [ ]:
selected.repeat_interleave(2, dim=-1)

tensor([[-0.9900, -0.9900,  0.9996,  0.9996,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.9602,  0.9602,  0.9982,  0.9982,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.2837,  0.2837,  0.9988,  0.9988,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,  1.0000,
          1.0000,  1.0000,

In [71]:
pos = 1
Ri = torch.diag(cos[pos, :].repeat_interleave(2))

In [77]:
sin_vec = sin[pos, :].repeat_interleave(2)
sin_vec[1::2]=0
torch.diag(sin_vec, diagonal=1)

tensor([[0.0000e+00, 8.4147e-01, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        ...,
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 1.0000e-38,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00]])

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [79]:


def apply_rope(q, k):
    
    
    B, S, D = q.shape
    pos = torch.arange(S, device=q.device).unsqueeze(1).float()
    dim = torch.arange(0, D, 2, device=q.device).float()
    freqs = 1.0 / (10000.0 ** (dim / D))
    angles = pos * freqs
    cos_a = torch.cos(angles)
    sin_a = torch.sin(angles)

    def rotate(x):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos_a - x2 * sin_a,
                            x1 * sin_a + x2 * cos_a], dim=-1).flatten(-2)

    return rotate(q), rotate(k)

In [90]:
import torch
max_seq_len = 10
theta = 0.1
d_k = 40

theta_mat = torch.arange(0, max_seq_len, 1, dtype = torch.float32).reshape((max_seq_len, 1)) @ (theta**(torch.arange(0, d_k, 2))).reshape(1, d_k//2)
cos = torch.cos(theta_mat)
sin = torch.sin(theta_mat)
cos_a = cos[positions, :]
sin_a = sin[positions, :]

In [94]:
x = torch.randn(1, 5, 40)
x1, x2 = x[..., 0::2], x[..., 1::2]
torch.stack([x1 * cos_a - x2 * sin_a,
                            x1 * sin_a + x2 * cos_a], dim=-1).flatten(-2).shape

torch.Size([1, 5, 40])

In [80]:
# Demo
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
# qr, kr = apply_rope(q, k)
B, S, D = q.shape
pos = torch.arange(S, device=q.device).unsqueeze(1).float()
dim = torch.arange(0, D, 2, device=q.device).float()
freqs = 1.0 / (10000.0 ** (dim / D))
angles = pos * freqs
cos_a = torch.cos(angles)
sin_a = torch.sin(angles)

In [99]:
import torch

def softmax(x: torch.Tensor, dim:int):
    max_x = torch.max(x, dim = dim, keepdim=True).values

    exps = torch.exp(x - max_x)
    denom = torch.sum(exps, dim = dim, keepdim=True)

    return  exps/ denom




In [102]:

x = torch.normal(mean = 0, std = 1, size = (2, 4, 8))
softmax(x, 2) - torch.softmax(x, dim = 2)

tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          -7.4506e-09,  0.0000e+00,  0.0000e+00],
         [ 5.9605e-08,  1.4901e-08,  7.4506e-09,  3.7253e-09,  7.4506e-09,
           0.0000e+00,  7.4506e-09,  0.0000e+00],
         [ 0.0000e+00, -3.7253e-09,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          -1.8626e-09,  0.0000e+00,  0.0000e+00],
         [-7.4506e-09,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00]],

        [[ 0.0000e+00,  0.0000e+00,  1.4901e-08,  0.0000e+00,  0.0000e+00,
           7.4506e-09,  1.4901e-08,  0.0000e+00],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00],
         [ 3.7253e-09,  1.4901e-08,  2.9802e-08,  0.0000e+00,  0.0000e+00,
           7.4506e-09,  0.0000e+00,  3.7253e-09],
         [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00]

In [188]:

inputs = torch.tensor([[1.0, 2.0, 3.0],
                    [1.0, 1.0, 1.0]])
targets = [2, 0]
logit = torch.softmax(inputs, dim = -1)
print(logit)
-torch.log(logit[[0, 1], targets]).mean()

tensor([[0.0900, 0.2447, 0.6652],
        [0.3333, 0.3333, 0.3333]])


tensor(0.7531)

In [203]:
def cross_entropy(logit, targets):
    '''
    logit: [batch_size, vocab_size]
    target: batch_size
        each element is a number between 0 to vocab_size - 1

    '''
    batch, _ = logit.shape
    logit_xi = logit[list(torch.arange(batch)), targets]
    M = torch.max(logit, dim = -1, keepdim=True).values
    M_flat = torch.max(logit, dim = -1).values
    return -(logit_xi - M_flat).sum() / batch + torch.log(torch.exp(logit - M).sum(dim = -1)).sum() / batch


In [204]:

cross_entropy(inputs, targets)

tensor(0.7531)

In [222]:
from collections.abc import Callable, Iterable
from typing import Optional
import math
import torch
from torch.optim.optimizer import Optimizer
class SGD(Optimizer):
    def __init__(self, params, lr = 1e-3):
        if lr<0:
            raise ValueError("fInvalid learing rate: {lr}")
        defaults = {"lr":lr}
        super().__init__(params, defaults)

    def step(self, closure:Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group['lr'] # -> group's corresponding learning rate
            for p in group['params']:
                if p.grad is None: # if no grad, skip, otherwise update one gradient step
                    continue

                state = self.state[p] # get state associate with p
                t = state.get("t", 0) # get iteration number from the state, or 0
                grad = p.grad.data # get the gradient of loss w.r.t. p
                p.data -= lr / math.sqrt(t+1) * grad # gradient descent update in-place
                state["t"] = t+1 # Increment iteration number 
        return loss


In [226]:
# from SGD import SGD
weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr = 1e2)
for t in range(100):
    opt.zero_grad() 
    loss = (weights**2).mean()
    print(loss.cpu().item())
    loss.backward()
    opt.step()

19.628950119018555
19.628950119018555
3.367795467376709
0.08059892803430557
1.0255685004686426e-16
1.1430596919838323e-18
3.849084324593485e-20
2.292926078744953e-21
1.967019997112034e-22
2.18557794987135e-23
2.9524746403513385e-24
4.652833753195152e-25
8.311487184917574e-26
1.648100837288173e-26
3.570929110018572e-27
8.35137053628251e-28
2.0878426340706277e-28
5.535948402632572e-29
1.5468169542922385e-29
4.530060320046362e-30
1.3842634640819987e-30
4.396485229420051e-31
1.4465108035244253e-31
4.916044622402283e-32
1.721451673504142e-32
6.197226788686239e-33
2.289141535743999e-33
8.660915643873916e-34
3.3511527895006893e-34
1.3242070135820286e-34
5.337039948236444e-35
2.1914476294370675e-35
9.157913163074767e-36
3.891211312782393e-36
1.679651673813485e-36
7.359600701182579e-37
3.2709341930334344e-37
1.4735954878389116e-37
6.725158673616063e-38
3.1073698158107244e-38
1.4528334574279051e-38
6.869955404454132e-39
3.284007410874568e-39
1.5862712629141572e-39
7.739203262650246e-40
3.8123585

In [41]:
import numpy as np
import torch
def data_loading(dataset:np.array, batch_size:int, context_length:int, device=None):
    # x: (n, )
    # batch sampling:
    # Given a randomly chosen starting index $i$, the input is $x[i:i+m]$ and the target/label is $x[i+1:i+m+1]$ 
    # the same window shifted by one position (the "next token" prediction target). 
    input_idx = np.random.randint(low = 0, high = len(dataset)-context_length, # here [low, high), right exclusive
                                      size=(batch_size, 1) )+ np.arange(context_length)
    print(input_idx)
    label_idx = input_idx + 1
    
    return dataset[input_idx], dataset[label_idx]




In [42]:
data_set =  np.array([10, 23, 5, 88, 47, 12, 33, 91, 6, 77])
batch_size = 2
context_length = 3
data_loading(data_set, batch_size, context_length, device = 'mps')

[[4 5 6]
 [6 7 8]]


(array([[47, 12, 33],
        [33, 91,  6]]),
 array([[12, 33, 91],
        [91,  6, 77]]))

In [40]:
start_indices = np.random.randint(low = 0, high = len(data_set)-context_length, size=(batch_size, 1))
batch_idx = start_indices + np.arange(context_length)

data_set[batch_idx] 

array([[88, 47, 12],
       [10, 23,  5]])

In [20]:
file_path = 'test/test_mmmp_dataset.npy'
np.save(
    file_path, data_set
)

In [21]:
data_mmp = np.load(file_path, mmap_mode='r')

In [22]:
data_loading(data_mmp, 2, 3)


tensor([[33, 91,  6],
        [33, 91,  6]])

In [44]:

import yaml

In [50]:
config_path = 'test/base.yaml'
with open(config_path) as f:
    base = yaml.safe_load(f)

In [55]:
base['model'], base['data']

({'vocab_size': 32000,
  'context_length': 256,
  'd_model': 512,
  'num_layers': 12,
  'num_heads': 8,
  'd_ff': 1344,
  'rope_theta': 10000.0},
 {'train_path': 'data/tokenized/train.bin',
  'val_path': 'data/tokenized/val.bin',
  'batch_size': 32})

In [57]:
base.keys()

dict_keys(['model', 'data', 'optim', 'lr_schedule', 'training', 'logging', 'resume_from'])